In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from math import sqrt

import dask.dataframe as dd
from dask.distributed import Client, LocalCluster

# plt.style.use('seaborn-v0_8')
# plotColors = (list(mcolors.TABLEAU_COLORS)*3)[:23]

In [2]:
%pip install distributed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: dask
    Found existing installation: dask 2026.1.2
    Uninstalling dask-2026.1.2:
      Successfully uninstalled dask-2026.1.2


Starting Dask Client

In [2]:
cluster = LocalCluster()
client = Client(cluster)
print(client)

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:40055
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:35209'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:41757'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:39851 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:39851
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:52050
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:44751 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:44751
INFO:distributed.core:Starting established connection to tcp://127

<Client: 'tcp://127.0.0.1:40055' processes=2 threads=2, memory=12.67 GiB>


In [3]:
# Downloading and loading dataset from drive
from google.colab import drive
drive.mount('/content/drive')

!wget -O "/content/drive/MyDrive/Crimes.csv" "https://data.cityofchicago.org/api/views/ijzp-q8t2/rows.csv?accessType=DOWNLOAD"

DATA_PATH = "/content/drive/MyDrive/Crimes.csv"

Mounted at /content/drive
--2026-04-01 18:05:19--  https://data.cityofchicago.org/api/views/ijzp-q8t2/rows.csv?accessType=DOWNLOAD
Resolving data.cityofchicago.org (data.cityofchicago.org)... 52.206.140.205, 52.206.140.199, 52.206.68.26
Connecting to data.cityofchicago.org (data.cityofchicago.org)|52.206.140.205|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/csv]
Saving to: ‘/content/drive/MyDrive/Crimes.csv’

/content/drive/MyDr     [   <=>              ]   1.50G   277KB/s    in 54m 45s 

2026-04-01 19:00:05 (478 KB/s) - ‘/content/drive/MyDrive/Crimes.csv’ saved [1607245895]



In [8]:

allCrimes = dd.read_csv(
    DATA_PATH,
    dtype={
        'X Coordinate': 'float64',
        'Y Coordinate': 'float64',
        'Ward': 'float64',
        'Community Area': 'float64',
        'District': 'float64',
        'FBI Code':'object',
        'IUCR':'object',
        'ID':'object'
    },
    assume_missing=True
)

print(allCrimes.dtypes)
print("Partitions:", allCrimes.npartitions)

ID                      string[pyarrow]
Case Number             string[pyarrow]
Date                    string[pyarrow]
Block                   string[pyarrow]
IUCR                    string[pyarrow]
Primary Type            string[pyarrow]
Description             string[pyarrow]
Location Description    string[pyarrow]
Arrest                             bool
Domestic                           bool
Beat                            float64
District                        float64
Ward                            float64
Community Area                  float64
FBI Code                string[pyarrow]
X Coordinate                    float64
Y Coordinate                    float64
Year                            float64
Updated On              string[pyarrow]
Latitude                        float64
Longitude                       float64
Location                string[pyarrow]
dtype: object
Partitions: 25


In [9]:
# Removing duplicates
print("Records before dedup:", len(allCrimes))
allCrimes = allCrimes.drop_duplicates(subset="ID")
print("Records after dedup:", len(allCrimes))

Records before dedup: 6823503


Records after dedup: 6823503


In [10]:
# Removing NaN values
#(removing entire row if a NaN value is present)
allCrimes = allCrimes.dropna()
print("Records after dropping NaN:", len(allCrimes))

Records after dropping NaN: 6162590


In [12]:
# Removing invalid coordinates
#(remove where coordinates are 0, they are clearly wring/missing)
allCrimes = allCrimes.reset_index(drop=True)

allCrimes = allCrimes[
    (allCrimes['X Coordinate'] != 0) &
    (allCrimes['Y Coordinate'] != 0) &
    (allCrimes['Longitude'] != 0) &
    (allCrimes['Latitude'] != 0)
]
print("Records after removing bad locations:", len(allCrimes))

Records after removing bad locations: 6162483


In [13]:
# convert date column from string to object
allCrimes['Date'] = dd.to_datetime(allCrimes['Date'], format='%m/%d/%Y %I:%M:%S %p')

In [14]:
# extracting time features from date
allCrimes['Hour']      = allCrimes['Date'].dt.hour
allCrimes['Month']     = allCrimes['Date'].dt.month
allCrimes['DayOfWeek'] = allCrimes['Date'].dt.dayofweek
allCrimes['Year']      = allCrimes['Date'].dt.year

In [15]:
# keep top 25 crime locations
# Compute value counts (this triggers execution)
loc_counts = allCrimes['Location Description'].value_counts().compute()
mostFrequentLocations = list(loc_counts[:25].index)

# Replace rare locations with 'OTHER'
allCrimes['Location Description'] = allCrimes['Location Description'].where(
    allCrimes['Location Description'].isin(mostFrequentLocations),
    other='OTHER'
)

# Same for Description column
desc_counts = allCrimes['Description'].value_counts().compute()
mostFrequentDescription = list(desc_counts[:25].index)

allCrimes['Description'] = allCrimes['Description'].where(
    allCrimes['Description'].isin(mostFrequentDescription),
    other='OTHER'
)

In [16]:
# convert to categorical for increasing speed
allCrimes['Primary Type']         = allCrimes['Primary Type'].astype('category')
allCrimes['Location Description'] = allCrimes['Location Description'].astype('category')
allCrimes['Description']          = allCrimes['Description'].astype('category')

In [17]:
# drop unnecessary columns
allCrimes = allCrimes.drop(
    columns=['Case Number', 'IUCR', 'Updated On', 'FBI Code', 'Beat', 'Ward'],
    errors='ignore'
)

In [18]:
# THIS is where all the above lazy steps actually execute (this is lazy evaluation of Dask)
allCrimes = allCrimes.compute()

print("Final clean dataset shape:", allCrimes.shape)
allCrimes.head()

Final clean dataset shape: (6162483, 19)


,ID,Date,Block,Primary Type,Description,Location Description,Arrest,Domestic,District,Community Area,X Coordinate,Y Coordinate,Year,Latitude,Longitude,Location,Hour,Month,DayOfWeek
0,13204489,2023-09-06 11:00:00,0000X E 8TH ST,THEFT,OTHER,OTHER,False,False,1.0,32.0,1176857.0,1896680.0,2023,41.871835,-87.626151,"(41.871834768, -87.62615082)",11,9,2
1,6272641,2008-05-27 01:00:00,105XX S PERRY AVE,ROBBERY,OTHER,STREET,False,True,5.0,49.0,1177463.0,1835161.0,2008,41.703007,-87.625785,"(41.703006756, -87.625784664)",1,5,1
2,13183567,2023-08-21 14:37:00,022XX N LEAMINGTON AVE,DECEPTIVE PRACTICE,OTHER,STREET,True,False,25.0,19.0,1141688.0,1914409.0,2023,41.921208,-87.754833,"(41.921207741, -87.754832585)",14,8,0
3,13201867,2023-09-06 15:10:00,069XX S VERNON AVE,ASSAULT,AGGRAVATED - HANDGUN,ALLEY,False,True,3.0,69.0,1180444.0,1859195.0,2023,41.768891,-87.614134,"(41.768891175, -87.614133593)",15,9,2
4,13202013,2023-09-06 10:30:00,051XX W HARRISON ST,ASSAULT,OTHER,OTHER,False,True,15.0,25.0,1142125.0,1896831.0,2023,41.872964,-87.753663,"(41.872963554, -87.753663295)",10,9,2


In [19]:
# creating a backup
backup = allCrimes.copy()